In [1]:
from srm.qa_flags import (
    ATTRS_TIME_INVARIANT,
    ATTRS_TIME_VARYING,
    FLAG_LIST_TIME_INVARIANT,
    FLAG_LIST_TIME_VARYING,
    combine_intermediate_flags,
    discover_leaves,
    parse_tag,
    write_final_qa_flags,
)

# A. Define what data arrays exist to traverse

In [2]:
# --- Run parameters --------------------------------------------------------
# This regional South-Africa-box run covers three GCMs, each written to its own icechunk store
# (same bucket/branch, named by GCM the same way srm.cache.ArtifactCache names pipeline output
# stores). Looping over GCMS -- rather than hardcoding one, as this notebook used to -- is what
# lets every leaf be compared against ITS OWN GCM's catalog and lineage instead of silently
# reusing whichever GCM happened to be hardcoded.


VARIABLES = ["tas", "tasmax", "tasmin", "pr", "rsds"]  # , "hurs"]

In [3]:
GCMS = ["CESM2-WACCM"]

# BRANCH = "full-regional-run-issue-534"
# ROOT_DIR = "s3://carbonplan-scratch/srm/output/qa/"
# STORE_SUBSET_BOUNDS = (-38.0, -19.0, 13.0, 36.0)
# STORE_SUBSET_ID = ArtifactCache._get_subset_id(STORE_SUBSET_BOUNDS)

BRANCH = "v0.13.0"
ROOT_DIR = "s3://us-west-2.opendata.source.coop/carbonplan/srm-downscaling/output/production/"
STORE_SUBSET_ID = "global"

In [4]:
[_, tags, _, _, _, tags_np] = discover_leaves(
    gcms=GCMS, branch=BRANCH, root_dir=ROOT_DIR, store_subset_id=STORE_SUBSET_ID
)

opened 1/1 stores on branch 'v0.13.0': CESM2-WACCM
71 leaves across 1 GCMs


In [5]:
FINAL_FLAG_DIR = "s3://carbonplan-scratch/srm/qaqc/flags/" + STORE_SUBSET_ID + "/"

In [6]:
BUCKET = "carbonplan-srm"
PREFIX = "scratch/output/qa-intermediate-flags"

# Write out final overall flags

In [7]:
tag = "CESM2-WACCM_rsds_ssp245_003"

[overall_flag_time_varying, overall_flag_time_invariant] = combine_intermediate_flags(
    tag=tag,
    flag_list_time_varying=FLAG_LIST_TIME_VARYING,
    flag_list_time_invariant=FLAG_LIST_TIME_INVARIANT,
    bucket=BUCKET,
    prefix=PREFIX,
)

lat = overall_flag_time_invariant.lat
lon = overall_flag_time_invariant.lon

In [16]:
print(len(tags))
# This loop takes about 15 minutes to run on v0.13.0 (31 global data arrays)

for i, tag in enumerate(tags):
    print(tag)
    [gcm, var, scenario, ens] = parse_tag(tag)
    # Get flags
    print("calculating flags")
    [overall_flag_time_varying, overall_flag_time_invariant] = combine_intermediate_flags(
        tag=tag,
        flag_list_time_varying=FLAG_LIST_TIME_VARYING,
        flag_list_time_invariant=FLAG_LIST_TIME_INVARIANT,
        bucket=BUCKET,
        prefix=PREFIX,
    )

    overall_flag_time_varying.compute()
    overall_flag_time_invariant.compute()

    # TO DO: write these flags out to source coop datasets
    # For now, writing these out to the same datasets where the individual flags are
    fpath_out = f"{FINAL_FLAG_DIR}{gcm}_{var}_{scenario}_{ens}.zarr"

    print("writing out flags")
    write_final_qa_flags(
        flag_data=overall_flag_time_varying,
        flag_name="flag_time_varying",
        dataset_path=fpath_out,
        attrs=ATTRS_TIME_VARYING,
        write_mode="a",
        time_varying=True,
    )

    write_final_qa_flags(
        flag_data=overall_flag_time_invariant,
        flag_name="flag_time_invariant",
        dataset_path=fpath_out,
        attrs=ATTRS_TIME_INVARIANT,
        write_mode="a",
        time_varying=False,
    )

31
CESM2-WACCM_tasmin_g6_1p5k_002
calculating flags
writing out flags
CESM2-WACCM_tasmin_g6_1p5k_003
calculating flags
writing out flags
CESM2-WACCM_pr_g6_1p5k_002
calculating flags
writing out flags
CESM2-WACCM_pr_g6_1p5k_003
calculating flags
writing out flags
CESM2-WACCM_rsds_g6_1p5k_002
calculating flags
writing out flags
CESM2-WACCM_rsds_g6_1p5k_003
calculating flags
writing out flags
CESM2-WACCM_tasmax_g6_1p5k_002
calculating flags
writing out flags
CESM2-WACCM_tasmax_g6_1p5k_003
calculating flags
writing out flags
CESM2-WACCM_tas_g6_1p5k_002
calculating flags
writing out flags
CESM2-WACCM_tas_g6_1p5k_003
calculating flags
writing out flags
CESM2-WACCM_pr_g6_1p5k_end_002
calculating flags
writing out flags
CESM2-WACCM_rsds_g6_1p5k_end_002
calculating flags
writing out flags
CESM2-WACCM_tasmax_g6_1p5k_end_002
calculating flags
writing out flags
CESM2-WACCM_tasmin_g6_1p5k_end_002
calculating flags
writing out flags
CESM2-WACCM_tas_g6_1p5k_end_002
calculating flags
writing out flags